# 01 – EDA Overview: AIO OIA Datathon 2026

**Mục tiêu:** Khám phá toàn bộ dataset, tìm patterns, và trả lời 10 MCQ câu hỏi.

- Training: `sales.csv` (2012-07-04 → 2022-12-31, 3833 ngày)
- Predict: Revenue + COGS cho 2023-01-01 → 2024-07-01 (548 rows)
- Supporting files: orders, products, customers, promotions, web_traffic, inventory, payments, returns, reviews


## Section 1: Setup & Load Data


In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, '..')

RAW = "../src/data/raw"
os.makedirs("../report/figures", exist_ok=True)
os.makedirs("../submissions", exist_ok=True)

print("Libraries loaded successfully.")
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}")


The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


E:\temp\ipykernel_53288\757461678.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Libraries loaded successfully.
NumPy: 1.26.4, Pandas: 2.2.0


In [3]:
# Load all CSVs
sales      = pd.read_csv(f"{RAW}/sales.csv",      parse_dates=["Date"])
sales      = sales.sort_values("Date").reset_index(drop=True)

orders     = pd.read_csv(f"{RAW}/orders.csv",     parse_dates=["order_date"])
products   = pd.read_csv(f"{RAW}/products.csv")
customers  = pd.read_csv(f"{RAW}/customers.csv",  parse_dates=["signup_date"])
promotions = pd.read_csv(f"{RAW}/promotions.csv", parse_dates=["start_date", "end_date"])
web        = pd.read_csv(f"{RAW}/web_traffic.csv",parse_dates=["date"])
inventory  = pd.read_csv(f"{RAW}/inventory.csv")
payments   = pd.read_csv(f"{RAW}/payments.csv")
returns    = pd.read_csv(f"{RAW}/returns.csv",    parse_dates=["return_date"])
reviews    = pd.read_csv(f"{RAW}/reviews.csv",    parse_dates=["review_date"])

print("Sales:", sales.shape, "→", sales["Date"].min().date(), "đến", sales["Date"].max().date())
print("Columns:", sales.columns.tolist())
print(sales.head())


Sales: (3833, 3) → 2012-07-04 đến 2022-12-31
Columns: ['Date', 'Revenue', 'COGS']
        Date     Revenue        COGS
0 2012-07-04  5123547.94  3982991.19
1 2012-07-05  2751773.45  2150580.23
2 2012-07-06  3054029.42  2517632.84
3 2012-07-07  2667930.94  2108246.62
4 2012-07-08  2360851.90  1808622.79


In [4]:
# Quick overview of all datasets
datasets = {
    "sales": sales, "orders": orders, "products": products,
    "customers": customers, "promotions": promotions, "web": web,
    "inventory": inventory, "payments": payments, "returns": returns, "reviews": reviews
}
for name, df in datasets.items():
    print(f"{name:12s}: {df.shape[0]:>8,} rows x {df.shape[1]:>2} cols | cols: {list(df.columns)[:5]}...")


sales       :    3,833 rows x  3 cols | cols: ['Date', 'Revenue', 'COGS']...
orders      :  646,945 rows x  8 cols | cols: ['order_id', 'order_date', 'customer_id', 'zip', 'order_status']...
products    :    2,412 rows x  8 cols | cols: ['product_id', 'product_name', 'category', 'segment', 'size']...
customers   :  121,930 rows x  7 cols | cols: ['customer_id', 'zip', 'city', 'signup_date', 'gender']...
promotions  :       50 rows x 10 cols | cols: ['promo_id', 'promo_name', 'promo_type', 'discount_value', 'start_date']...
web         :    3,652 rows x  7 cols | cols: ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate']...
inventory   :   60,247 rows x 17 cols | cols: ['snapshot_date', 'product_id', 'stock_on_hand', 'units_received', 'units_sold']...
payments    :  646,945 rows x  4 cols | cols: ['order_id', 'payment_method', 'payment_value', 'installments']...
returns     :   39,939 rows x  7 cols | cols: ['return_id', 'order_id', 'product_id', 'return_date', 'return_r

## Section 2: Basic Stats – Sales


In [5]:
print("=== SALES BASIC STATS ===")
print(sales[["Revenue", "COGS"]].describe())
print("\nNull values:", sales.isnull().sum().to_dict())

# COGS ratio & gross margin
sales["cogs_ratio"] = np.where(sales["Revenue"] > 0, sales["COGS"] / sales["Revenue"], np.nan)
sales["gross_margin"] = np.where(sales["Revenue"] > 0, (sales["Revenue"] - sales["COGS"]) / sales["Revenue"], np.nan)
print(f"Rows with zero Revenue: {(sales["Revenue"] <= 0).sum()}")

print(f"\nCOGS/Revenue ratio: mean={sales['cogs_ratio'].mean():.4f}, std={sales['cogs_ratio'].std():.4f}")
print(f"Gross margin:       mean={sales['gross_margin'].mean():.4f}")
print(f"Min Revenue:  {sales['Revenue'].min():,.2f}")
print(f"Max Revenue:  {sales['Revenue'].max():,.2f}")
print(f"Total Revenue (all years): {sales['Revenue'].sum():,.2f}")


=== SALES BASIC STATS ===
            Revenue          COGS
count  3.833000e+03  3.833000e+03
mean   4.286584e+06  3.695134e+06
std    2.624840e+06  2.219789e+06
min    2.798139e+05  2.365763e+05
25%    2.471089e+06  2.150580e+06
50%    3.647304e+06  3.161113e+06
75%    5.350877e+06  4.637294e+06
max    2.090527e+07  1.653586e+07

Null values: {'Date': 0, 'Revenue': 0, 'COGS': 0}
Rows with zero Revenue: 0

COGS/Revenue ratio: mean=0.8746, std=0.1274
Gross margin:       mean=0.1254
Min Revenue:  279,813.94
Max Revenue:  20,905,271.35
Total Revenue (all years): 16,430,476,585.53


## Section 3: Time Series Plot (Revenue & COGS)


In [6]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(sales["Date"], sales["Revenue"], lw=0.6, alpha=0.8, color='steelblue')
axes[0].set_title("Daily Revenue 2012–2022", fontsize=14)
axes[0].set_ylabel("Revenue")

axes[1].plot(sales["Date"], sales["COGS"], lw=0.6, alpha=0.8, color='orange')
axes[1].set_title("Daily COGS 2012–2022", fontsize=14)
axes[1].set_ylabel("COGS")

plt.tight_layout()
plt.savefig("../report/figures/01_timeseries.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved → ../report/figures/01_timeseries.png")


Figure saved → ../report/figures/01_timeseries.png


## Section 4: YoY Growth Analysis


In [7]:
sales["year"]        = sales["Date"].dt.year
sales["month"]       = sales["Date"].dt.month
sales["day_of_week"] = sales["Date"].dt.dayofweek  # 0=Mon, 6=Sun

annual = sales.groupby("year")[["Revenue", "COGS"]].sum()
annual["rev_yoy_pct"]  = annual["Revenue"].pct_change() * 100
annual["cogs_yoy_pct"] = annual["COGS"].pct_change() * 100

print("=== ANNUAL TOTALS & YoY GROWTH ===")
print(annual.round(2))

avg_growth = annual["rev_yoy_pct"].dropna().mean()
print(f"\nAverage YoY Revenue Growth: {avg_growth:.2f}%")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(annual.index, annual["Revenue"] / 1e6, color='steelblue', label='Revenue (M)')
ax.bar(annual.index, annual["COGS"] / 1e6, color='orange', alpha=0.7, label='COGS (M)')
ax.set_title("Annual Revenue & COGS (Millions)", fontsize=14)
ax.set_xlabel("Year"); ax.set_ylabel("Amount (M)")
ax.legend()
plt.tight_layout()
plt.savefig("../report/figures/03_annual_revenue.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved → ../report/figures/03_annual_revenue.png")


=== ANNUAL TOTALS & YoY GROWTH ===
           Revenue          COGS  rev_yoy_pct  cogs_yoy_pct
year                                                       
2012  7.414977e+08  5.874619e+08          NaN           NaN
2013  1.657169e+09  1.465980e+09       123.49        149.54
2014  1.871846e+09  1.574607e+09        12.95          7.41
2015  1.889934e+09  1.665442e+09         0.97          5.77
2016  2.104641e+09  1.780559e+09        11.36          6.91
2017  1.911164e+09  1.694386e+09        -9.19         -4.84
2018  1.850122e+09  1.542176e+09        -3.19         -8.98
2019  1.136801e+09  1.005203e+09       -38.56        -34.82
2020  1.054512e+09  8.860851e+08        -7.24        -11.85
2021  1.043040e+09  9.411301e+08        -1.09          6.21
2022  1.169749e+09  1.020420e+09        12.15          8.42

Average YoY Revenue Growth: 10.17%
Figure saved → ../report/figures/03_annual_revenue.png


## Section 5: Seasonality Analysis


In [8]:
# Monthly pattern
monthly_avg = sales.groupby("month")["Revenue"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(monthly_avg.index, monthly_avg.values, color='steelblue')
axes[0].set_title("Average Revenue by Month")
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(
    ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],
    rotation=45
)

# Day of week pattern
dow_avg = sales.groupby("day_of_week")["Revenue"].mean()
axes[1].bar(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], dow_avg.values, color='coral')
axes[1].set_title("Average Revenue by Day of Week")

plt.tight_layout()
plt.savefig("../report/figures/02_seasonality.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved → ../report/figures/02_seasonality.png")

best_month = monthly_avg.idxmax()
best_dow   = dow_avg.idxmax()
dow_names  = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
month_names= ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
print(f"Best month:        {month_names[best_month-1]} (month {best_month}) – avg {monthly_avg[best_month]:,.2f}")
print(f"Best day of week:  {dow_names[best_dow]} (dow {best_dow}) – avg {dow_avg[best_dow]:,.2f}")


Figure saved → ../report/figures/02_seasonality.png
Best month:        May (month 5) – avg 6,575,416.35
Best day of week:  Wed (dow 2) – avg 4,680,064.84


## Section 6: MCQ Answers (10 câu)

Mỗi câu được phân tích từ data thực tế, in ra answer kèm evidence cụ thể.


In [9]:
print("=" * 70)
print("MCQ ANALYSIS – 10 QUESTIONS")
print("=" * 70)

# ── Q1: YoY Growth Trend ──────────────────────────────────────────────────
avg_yoy = annual["rev_yoy_pct"].dropna().mean()
pos_yoy = (annual["rev_yoy_pct"].dropna() > 0).sum()
total_y = len(annual["rev_yoy_pct"].dropna())
trend   = "positive (upward)" if avg_yoy > 0 else "negative (downward)"
print(f"\nQ1: What is the overall YoY revenue growth trend (2012–2022)?")
print(f"    → ANSWER: {trend.upper()}")
print(f"       Evidence: avg YoY = {avg_yoy:.2f}%, positive in {pos_yoy}/{total_y} years")


MCQ ANALYSIS – 10 QUESTIONS

Q1: What is the overall YoY revenue growth trend (2012–2022)?
    → ANSWER: POSITIVE (UPWARD)
       Evidence: avg YoY = 10.17%, positive in 5/10 years


In [10]:
# ── Q2: Which month has highest average revenue? ──────────────────────────
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
best_month  = monthly_avg.idxmax()
worst_month = monthly_avg.idxmin()
print(f"Q2: Which month has the highest average daily revenue?")
print(f"    → ANSWER: {month_names[best_month-1].upper()} (month {best_month})")
print(f"       Evidence: avg = {monthly_avg[best_month]:,.2f} | lowest: {month_names[worst_month-1]} = {monthly_avg[worst_month]:,.2f}")
print(f"\n    Monthly averages:")
for m, v in monthly_avg.items():
    bar = '█' * int(v / monthly_avg.max() * 20)
    print(f"      {month_names[m-1]:3s}: {v:>12,.0f}  {bar}")


Q2: Which month has the highest average daily revenue?
    → ANSWER: MAY (month 5)
       Evidence: avg = 6,575,416.35 | lowest: Dec = 2,524,349.62

    Monthly averages:
      Jan:    2,591,155  ███████
      Feb:    3,480,801  ██████████
      Mar:    4,928,185  ██████████████
      Apr:    6,532,952  ███████████████████
      May:    6,575,416  ████████████████████
      Jun:    6,427,109  ███████████████████
      Jul:    4,659,789  ██████████████
      Aug:    4,441,193  █████████████
      Sep:    3,797,826  ███████████
      Oct:    3,302,725  ██████████
      Nov:    2,611,295  ███████
      Dec:    2,524,350  ███████


In [11]:
# ── Q3: Which day of week has highest revenue? ────────────────────────────
dow_names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
best_dow  = dow_avg.idxmax()
worst_dow = dow_avg.idxmin()
print(f"Q3: Which day of week has the highest average revenue?")
print(f"    → ANSWER: {dow_names[best_dow].upper()} (dow index {best_dow})")
print(f"       Evidence: avg = {dow_avg[best_dow]:,.2f} | lowest: {dow_names[worst_dow]} = {dow_avg[worst_dow]:,.2f}")
print(f"\n    DoW averages:")
for d, v in dow_avg.items():
    bar = '█' * int(v / dow_avg.max() * 20)
    print(f"      {dow_names[d]:3s}: {v:>12,.0f}  {bar}")


Q3: Which day of week has the highest average revenue?
    → ANSWER: WED (dow index 2)
       Evidence: avg = 4,680,064.84 | lowest: Sat = 3,906,580.84

    DoW averages:
      Mon:    4,311,035  ██████████████████
      Tue:    4,465,103  ███████████████████
      Wed:    4,680,065  ████████████████████
      Thu:    4,523,044  ███████████████████
      Fri:    4,046,390  █████████████████
      Sat:    3,906,581  ████████████████
      Sun:    4,073,854  █████████████████


In [12]:
# ── Q4: Average COGS/Revenue ratio (gross margin) ─────────────────────────
mean_cogs_ratio   = sales["cogs_ratio"].mean()
mean_gross_margin = sales["gross_margin"].mean()
print(f"Q4: What is the average COGS/Revenue ratio and gross margin?")
print(f"    → ANSWER: COGS/Revenue = {mean_cogs_ratio:.4f} ({mean_cogs_ratio*100:.2f}%)")
print(f"              Gross Margin = {mean_gross_margin:.4f} ({mean_gross_margin*100:.2f}%)")
print(f"       Evidence: std of ratio = {sales['cogs_ratio'].std():.4f} (very stable)")
print(f"                 min={sales['cogs_ratio'].min():.4f}, max={sales['cogs_ratio'].max():.4f}")


Q4: What is the average COGS/Revenue ratio and gross margin?
    → ANSWER: COGS/Revenue = 0.8746 (87.46%)
              Gross Margin = 0.1254 (12.54%)
       Evidence: std of ratio = 0.1274 (very stable)
                 min=0.7131, max=1.5746


In [13]:
# ── Q5: Web Traffic Trend ─────────────────────────────────────────────────
web["year"] = web["date"].dt.year
web_annual  = web.groupby("year")["sessions"].sum()
web_yoy     = web_annual.pct_change() * 100
web_avg_growth = web_yoy.dropna().mean()

print(f"Q5: What is the web traffic (sessions) trend over time?")
print(f"    → ANSWER: {'GROWING' if web_avg_growth > 0 else 'DECLINING'} – avg YoY sessions growth = {web_avg_growth:.2f}%")
print(f"       Evidence:")
for yr, sessions, yoy in zip(web_annual.index, web_annual.values, web_yoy.values):
    yoy_str = f"{yoy:+.1f}%" if not np.isnan(yoy) else "base"
    print(f"         {yr}: {sessions:>12,} sessions  {yoy_str}")


Q5: What is the web traffic (sessions) trend over time?
    → ANSWER: GROWING – avg YoY sessions growth = 5.58%
       Evidence:
         2013:    6,801,940 sessions  base
         2014:    7,340,960 sessions  +7.9%
         2015:    7,861,938 sessions  +7.1%
         2016:    8,403,399 sessions  +6.9%
         2017:    8,992,602 sessions  +7.0%
         2018:    9,415,085 sessions  +4.7%
         2019:    9,990,148 sessions  +6.1%
         2020:   10,591,082 sessions  +6.0%
         2021:   10,991,725 sessions  +3.8%
         2022:   11,063,658 sessions  +0.7%


In [ ]:
# ── Q6: Return Rate ───────────────────────────────────────────────────────
# Use unique order IDs for accurate rate
if "order_id" in returns.columns and "order_id" in orders.columns:
    returned_orders = returns["order_id"].nunique()
    total_orders_unique = orders["order_id"].nunique()
    return_rate = returned_orders / total_orders_unique * 100
    print(f"Orders with returns: {returned_orders:,} / {total_orders_unique:,} = {return_rate:.2f}%")
else:
    # Fallback to row count ratio
    return_rate = len(returns) / len(orders) * 100
    print(f"Return rate (row ratio): {return_rate:.2f}%")

# Top return reasons
top_reasons = returns["return_reason"].value_counts().head(5)

print(f"Q6: What is the return rate (returns / orders)?")
print(f"    → ANSWER: {return_rate:.2f}%")
print(f"Top return reasons:")
print(top_reasons.to_string())


SyntaxError: unterminated f-string literal (detected at line 18) (3604943766.py, line 18)

In [ ]:
# ── Q7: Top Payment Method ────────────────────────────────────────────────
# payments.csv has payment_method; orders.csv also has payment_method
pay_counts = payments["payment_method"].value_counts()
top_payment = pay_counts.index[0]

pay_pcts = pay_counts / len(payments) * 100
print(f"Q7: What is the most popular payment method?")
print(f"    → ANSWER: {top_payment.upper()}")
print(f"       Evidence (from payments.csv):")
for method, cnt in pay_counts.items():
    pct = pay_pcts[method]
    bar = '█' * int(pct / pay_pcts.max() * 20)
    print(f"  {method:20s}: {cnt:>8,}  ({pct:.1f}%)  {bar}")


In [ ]:
# ── Q8: Promotion Impact – count promos by year ───────────────────────────
promotions["start_year"] = promotions["start_date"].dt.year
promos_by_year = promotions.groupby("start_year").size()

# Also check promo types
promo_types = promotions["promo_type"].value_counts()

print(f"Q8: How are promotions distributed over the years? What are the main types?")
print(f"    → ANSWER: {len(promotions)} total promotions across {promotions['start_year'].nunique()} years")
print(f"       Promos by year:")
print(promos_by_year.to_string())
print(f"\n       Promo types:")
print(promo_types.to_string())

# Trend
if len(promos_by_year) > 1:
    trend = "increasing" if promos_by_year.iloc[-1] > promos_by_year.iloc[0] else "decreasing"
    print(f"\n       Trend: promotions are {trend} over time")


In [ ]:
# ── Q9: Customer Growth Trend ─────────────────────────────────────────────
customers["signup_year"] = customers["signup_date"].dt.year
cust_by_year = customers.groupby("signup_year").size()
cust_yoy     = cust_by_year.pct_change() * 100
avg_cust_growth = cust_yoy.dropna().mean()

print(f"Q9: What is the customer signup growth trend?")
print(f"    → ANSWER: {'GROWING' if avg_cust_growth > 0 else 'DECLINING'} – avg YoY growth = {avg_cust_growth:.2f}%")
print(f"       Evidence:")
for yr in cust_by_year.index:
    yoy_val = cust_yoy[yr]
    yoy_str = f"{yoy_val:+.1f}%" if not np.isnan(yoy_val) else "base"
    print(f"         {yr}: {cust_by_year[yr]:>8,} new customers  {yoy_str}")

# Acquisition channels
acq_channels = customers["acquisition_channel"].value_counts().head(5)
print(f"\n       Top acquisition channels:")
print(acq_channels.to_string())


In [ ]:
# ── Q10: Inventory – Stockout Rate ───────────────────────────────────────
stockout_rate  = inventory["stockout_flag"].mean() * 100
overstock_rate = inventory["overstock_flag"].mean() * 100
avg_fill_rate  = inventory["fill_rate"].mean() * 100

# Stockout by category
cat_stockout = inventory.groupby("category")["stockout_flag"].mean().sort_values(ascending=False) * 100

print(f"Q10: What is the stockout rate from inventory data?")
print(f"     → ANSWER: {stockout_rate:.2f}% of inventory snapshots show stockout")
print(f"        Evidence:")
print(f"          Stockout rate:   {stockout_rate:.2f}%")
print(f"          Overstock rate:  {overstock_rate:.2f}%")
print(f"          Avg fill rate:   {avg_fill_rate:.2f}%")
print(f"          Total snapshots: {len(inventory):,}")
print(f"\n        Stockout by category:")
print(cat_stockout.round(2).to_string())


In [ ]:
# ── Summary printout ──────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("QUICK REFERENCE – MCQ ANSWERS SUMMARY")
print("=" * 70)
dow_names_full   = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
month_names_full = ["January","February","March","April","May","June",
                    "July","August","September","October","November","December"]

print(f"Q1  YoY Revenue Trend:      {trend.upper()} (avg {avg_yoy:.1f}%/yr)")
print(f"Q2  Peak Revenue Month:     {month_names_full[best_month-1]}")
print(f"Q3  Peak Revenue DoW:       {dow_names_full[best_dow]}")
print(f"Q4  Avg COGS/Rev ratio:     {mean_cogs_ratio*100:.2f}% (gross margin {mean_gross_margin*100:.2f}%)")
print(f"Q5  Web Traffic Trend:      {'Growing' if web_avg_growth > 0 else 'Declining'} ({web_avg_growth:.1f}%/yr avg)")
print(f"Q6  Return Rate:            {return_rate:.2f}%")
print(f"Q7  Top Payment Method:     {top_payment}")
print(f"Q8  Total Promotions:       {len(promotions)} (main type: {promo_types.index[0]})")
print(f"Q9  Customer Growth Trend:  {'Growing' if avg_cust_growth > 0 else 'Declining'} ({avg_cust_growth:.1f}%/yr avg)")
print(f"Q10 Stockout Rate:          {stockout_rate:.2f}% (fill rate {avg_fill_rate:.2f}%)")


## Section 7: Outlier Detection


In [ ]:
# Phát hiện spike bất thường bằng z-score
sales["rev_z"]  = (sales["Revenue"] - sales["Revenue"].mean()) / sales["Revenue"].std()
sales["cogs_z"] = (sales["COGS"]    - sales["COGS"].mean())    / sales["COGS"].std()

outliers = sales[sales["rev_z"].abs() > 3]
print(f"Outlier days (|z| > 3): {len(outliers)}")
print(outliers[["Date", "Revenue", "COGS", "rev_z"]].head(20).to_string())


In [ ]:
# Plot outliers highlighted on time series
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(sales["Date"], sales["Revenue"], lw=0.5, alpha=0.7, color='steelblue', label='Revenue')
ax.scatter(outliers["Date"], outliers["Revenue"], color='red', s=20, zorder=5, label=f'Outliers (n={len(outliers)})')
ax.set_title("Revenue with Outliers Highlighted (|z| > 3)", fontsize=13)
ax.set_ylabel("Revenue")
ax.legend()
plt.tight_layout()
plt.savefig("../report/figures/04_outliers.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved → ../report/figures/04_outliers.png")


## Section 8: Rolling Averages & Trend Decomposition


In [ ]:
# 30-day and 90-day rolling mean
sales["rev_roll30"] = sales["Revenue"].rolling(30, center=True).mean()
sales["rev_roll90"] = sales["Revenue"].rolling(90, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(sales["Date"], sales["Revenue"],     lw=0.4, alpha=0.4, color='steelblue', label='Daily Revenue')
ax.plot(sales["Date"], sales["rev_roll30"],  lw=1.2, color='darkorange',   label='30-day MA')
ax.plot(sales["Date"], sales["rev_roll90"],  lw=2.0, color='darkred',      label='90-day MA')
ax.set_title("Revenue with 30-day & 90-day Moving Averages", fontsize=13)
ax.set_ylabel("Revenue")
ax.legend()
plt.tight_layout()
plt.savefig("../report/figures/05_moving_avg.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved → ../report/figures/05_moving_avg.png")


## Section 9: Sample Submission Preview


In [ ]:
submission = pd.read_csv(f"{RAW}/sample_submission.csv")
print("Sample submission shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print(submission.head(10))
print(f"\nDate range: {submission.iloc[:, 0].min()} → {submission.iloc[:, 0].max()}")


## Section 10: EDA Summary


In [ ]:
print("=" * 70)
print("EDA COMPLETE – KEY FINDINGS")
print("=" * 70)
print(f"  Training period:  {sales['Date'].min().date()} → {sales['Date'].max().date()} ({len(sales)} days)")
print(f"  Predict period:   2023-01-01 → 2024-07-01 ({len(submission)} rows)")
print(f"  Total Revenue:    {sales['Revenue'].sum():>20,.2f}")
print(f"  Total COGS:       {sales['COGS'].sum():>20,.2f}")
print(f"  Avg daily Rev:    {sales['Revenue'].mean():>20,.2f}")
print(f"  Avg daily COGS:   {sales['COGS'].mean():>20,.2f}")
print(f"  COGS/Rev ratio:   {mean_cogs_ratio:.4f}")
print(f"  Outlier days:     {len(outliers)} (|z|>3)")
print(f"  Figures saved to: ../report/figures/ (5 PNG files)")
print("\n  → Ready for feature engineering & model training")
